# FEATURE SELECTION - VN INDEX

## Import libraries

In [81]:
import os
import sys
import random
from datetime import datetime, timedelta
import pandas as pd
import lightning as L
import torch.nn as nn
import torch
from torch.optim import Adam
import torch.nn.functional as F
from torch.utils.data import TensorDataset, DataLoader
from tsfresh.utilities.dataframe_functions import (
    roll_time_series,
)
from tsfresh import extract_features, select_features
from tsfresh.utilities.dataframe_functions import impute
from tsfresh.feature_extraction import ComprehensiveFCParameters, EfficientFCParameters
import matplotlib.pylab as plt
from sklearn.preprocessing import MinMaxScaler
from lightning.pytorch.callbacks import EarlyStopping
from dataclasses import asdict
import ipynbname
from sklearn.inspection import permutation_importance
import xgboost as xgb
import seaborn as sns
from tqdm import tqdm
from sklearn.model_selection import train_test_split
from sklearn.base import clone

sys.path.append(os.path.abspath(os.path.join(os.getcwd(), "..")))

from logger.logger import Logger
from utils.constants import *
from dtos.config_dtos.config_dto import ConfigDto
from tabular_database_driver.postgre_sql_driver import PostgreSQLDriver
from dtos.tabular_database_driver_dtos.postgre_sql_connection_dto import (
    PostgreSQLConnectionDto,
)
from utils.enums import (
    LossFunctionType,
    ModelAchitectureType,
    OptimizerType,
    ScalerType,
)
from ta.ta_functions import *

In [82]:
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

## Helper functions

In [83]:
def get_weekends(from_date: str, to_date: str):
    start = datetime.strptime(from_date, "%Y-%m-%d")
    end = datetime.strptime(to_date, "%Y-%m-%d")

    weekends = []
    current = start

    while current <= end:
        if current.weekday() in (5, 6):  # 5 = Saturday, 6 = Sunday
            weekends.append(current.strftime("%Y-%m-%d"))
        current += timedelta(days=1)

    return weekends

## Parameters

In [84]:
STOCK_NAME = "vn_index"
NOTEBOOK_NAME = ipynbname.name()
RANDOM_SEED = 18
MAX_TIMESHIFT = 5
MIN_TIMESHIFT = 5
FORECAST_HORIZON = 5
COLUMN_ID = "stock"
TARGET_COLUMN = f"close_{FORECAST_HORIZON}"
DATE_COLUMN = "date"
FEATURE_COLUMNS = []  # all TA indicators

# Inclusive
TRAIN_RANGE = ("2000-01-01", "2021-12-31")
VALIDATION_RANGE = ("2022-01-01", "2023-12-31")
TEST_RANGE = ("2024-01-01", "2026-02-26")

In [85]:
WEEKENDS = get_weekends(TRAIN_RANGE[0], TEST_RANGE[1])
HOLIDAYS = []

DAYOFFS = []
DAYOFFS.extend(WEEKENDS)
DAYOFFS.extend(HOLIDAYS)

DAYOFFS[:10], DAYOFFS[-10:]

(['2000-01-01',
  '2000-01-02',
  '2000-01-08',
  '2000-01-09',
  '2000-01-15',
  '2000-01-16',
  '2000-01-22',
  '2000-01-23',
  '2000-01-29',
  '2000-01-30'],
 ['2026-01-24',
  '2026-01-25',
  '2026-01-31',
  '2026-02-01',
  '2026-02-07',
  '2026-02-08',
  '2026-02-14',
  '2026-02-15',
  '2026-02-21',
  '2026-02-22'])

## Load data

In [86]:
my_logger = Logger(file_name=f"{FEATURE_SELECTION_LOG_FILE_BASE}/vn_index/test")

In [87]:
my_connection_model = PostgreSQLConnectionDto(
    logger=my_logger,
    host=os.getenv("POSTGRES_HOST"),
    user=os.getenv("POSTGRES_USER"),
    password=os.getenv("POSTGRES_PASSWORD"),
    port=os.getenv("POSTGRES_PORT"),
    database=os.getenv("GOLD_POSTGRES_DATABASE"),
)

In [88]:
my_postgresql_driver = PostgreSQLDriver(logger=my_logger)
my_postgresql_driver.connect(my_connection_model)

<DatabaseExecutionStatus.SUCCESS: 'success'>

In [89]:
vn_index_df = my_postgresql_driver.select(
    schema_name="stock_market", table_name="vn_index"
)

In [90]:
dtype_map = {
    "date": str,
    "open": float,
    "high": float,
    "low": float,
    "close": float,
    "adjust": float,
    "change": float,
    "percent_change": float,
    "matching_volume": float,
    "matching_value": float,
    "negotiate_volume": float,
    "negotiate_value": float,
    "number_of_buy_orders": float,
    "buy_volume": float,
    "average_volume_per_buy_order": float,
    "number_of_sell_orders": float,
    "sell_volume": float,
    "average_volume_per_sell_order": float,
    "net_volume": float,
}

vn_index_df = (
    vn_index_df.astype(dtype_map)
    .dropna(subset=["close"])
    .sort_values(by=["date"])
    .reset_index(drop=True)
)

In [91]:
vn_index_df

,date,open,high,low,close,adjust,change,percent_change,matching_volume,matching_value,negotiate_volume,negotiate_value,number_of_buy_orders,buy_volume,average_volume_per_buy_order,number_of_sell_orders,sell_volume,average_volume_per_sell_order,net_volume
0,2008-03-07,640.140,640.140,640.140,640.140,640.140,28.970000,4.740000,8.169220e+06,5.152538e+11,350040.0,2.910304e+10,26816.0,5.268141e+07,1965.000000,3448.0,9.057040e+06,2627.0,4.362437e+07
1,2008-03-08,646.190,646.190,646.190,646.190,646.190,25.363333,4.106667,1.314382e+07,8.298321e+11,769057.0,4.594466e+10,27442.0,5.068024e+07,1852.333333,7486.0,1.712815e+07,2464.0,3.355209e+07
2,2008-03-09,652.240,652.240,652.240,652.240,652.240,21.756667,3.473333,1.811842e+07,1.144410e+12,1188073.0,6.278627e+10,28068.0,4.867907e+07,1739.666667,11523.0,2.519927e+07,2301.0,2.347980e+07
3,2008-03-10,658.290,658.290,658.290,658.290,658.290,18.150000,2.840000,2.309302e+07,1.458989e+12,1607090.0,7.962789e+10,28694.0,4.667790e+07,1627.000000,15561.0,3.327038e+07,2138.0,1.340752e+07
4,2008-03-11,638.710,638.710,638.710,638.710,638.710,-19.580000,-2.970000,1.384280e+07,8.374665e+11,226000.0,1.209960e+10,14464.0,1.933131e+07,1337.000000,17910.0,2.811427e+07,1570.0,-8.782960e+06
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6561,2026-02-22,1832.317,1859.236,1829.801,1856.535,1856.535,33.445000,1.837000,7.158068e+08,2.227241e+13,23581058.0,7.924402e+11,476711.0,1.233848e+09,2591.300000,393692.0,1.198445e+09,3047.9,3.540296e+07
6562,2026-02-23,1833.900,1863.130,1833.900,1860.140,1860.140,36.050000,1.980000,7.313611e+08,2.264621e+13,23192200.0,7.774309e+11,486603.0,1.253004e+09,2575.000000,401281.0,1.213944e+09,3025.0,3.906029e+07
6563,2026-02-24,1862.180,1867.690,1849.600,1867.620,1867.620,7.480000,0.400000,9.667547e+08,3.120594e+13,22413147.0,7.795734e+11,598087.0,1.575227e+09,2634.000000,507471.0,1.648273e+09,3248.0,-7.304628e+07
6564,2026-02-25,1869.490,1876.010,1855.890,1860.910,1860.910,-6.710000,-0.360000,1.083635e+09,3.567852e+13,62225450.0,1.856894e+12,694266.0,1.862364e+09,2682.000000,597096.0,1.892734e+09,3170.0,-3.036983e+07


## Data transformation

In [92]:
vn_index_df.shape

(6566, 19)

### Remove DAYOFFS

In [93]:
vn_index_df_t1 = vn_index_df[~vn_index_df["date"].isin(DAYOFFS)]
vn_index_df_t1

,date,open,high,low,close,adjust,change,percent_change,matching_volume,matching_value,negotiate_volume,negotiate_value,number_of_buy_orders,buy_volume,average_volume_per_buy_order,number_of_sell_orders,sell_volume,average_volume_per_sell_order,net_volume
0,2008-03-07,640.140,640.140,640.140,640.140,640.140,28.970,4.740,8.169220e+06,5.152538e+11,350040.0,2.910304e+10,26816.0,5.268141e+07,1965.0,3448.0,9.057040e+06,2627.0,4.362437e+07
3,2008-03-10,658.290,658.290,658.290,658.290,658.290,18.150,2.840,2.309302e+07,1.458989e+12,1607090.0,7.962789e+10,28694.0,4.667790e+07,1627.0,15561.0,3.327038e+07,2138.0,1.340752e+07
4,2008-03-11,638.710,638.710,638.710,638.710,638.710,-19.580,-2.970,1.384280e+07,8.374665e+11,226000.0,1.209960e+10,14464.0,1.933131e+07,1337.0,17910.0,2.811427e+07,1570.0,-8.782960e+06
5,2008-03-12,643.900,643.900,643.900,643.900,643.900,5.190,0.810,1.216691e+07,7.659728e+11,290070.0,2.036917e+10,16836.0,2.378011e+07,1412.0,12032.0,2.034621e+07,1691.0,3.433900e+06
6,2008-03-13,647.600,647.600,647.600,647.600,647.600,3.700,0.570,8.930090e+06,5.844466e+11,289160.0,1.702626e+10,13298.0,1.703506e+07,1281.0,12256.0,1.900562e+07,1551.0,-1.970560e+06
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6559,2026-02-20,1829.151,1851.448,1821.603,1849.325,1849.325,28.235,1.551,6.846983e+08,2.152480e+13,24358775.0,8.224590e+11,456926.0,1.195538e+09,2623.9,378513.0,1.167449e+09,3093.7,2.808830e+07
6562,2026-02-23,1833.900,1863.130,1833.900,1860.140,1860.140,36.050,1.980,7.313611e+08,2.264621e+13,23192200.0,7.774309e+11,486603.0,1.253004e+09,2575.0,401281.0,1.213944e+09,3025.0,3.906029e+07
6563,2026-02-24,1862.180,1867.690,1849.600,1867.620,1867.620,7.480,0.400,9.667547e+08,3.120594e+13,22413147.0,7.795734e+11,598087.0,1.575227e+09,2634.0,507471.0,1.648273e+09,3248.0,-7.304628e+07
6564,2026-02-25,1869.490,1876.010,1855.890,1860.910,1860.910,-6.710,-0.360,1.083635e+09,3.567852e+13,62225450.0,1.856894e+12,694266.0,1.862364e+09,2682.0,597096.0,1.892734e+09,3170.0,-3.036983e+07


### Create target

In [94]:
vn_index_df_t2 = vn_index_df_t1.copy()
vn_index_df_t2[f"close_{FORECAST_HORIZON}"] = vn_index_df_t1["close"].shift(
    -FORECAST_HORIZON
)
vn_index_df_t2

,date,open,high,low,close,adjust,change,percent_change,matching_volume,matching_value,negotiate_volume,negotiate_value,number_of_buy_orders,buy_volume,average_volume_per_buy_order,number_of_sell_orders,sell_volume,average_volume_per_sell_order,net_volume,close_5
0,2008-03-07,640.140,640.140,640.140,640.140,640.140,28.970,4.740,8.169220e+06,5.152538e+11,350040.0,2.910304e+10,26816.0,5.268141e+07,1965.0,3448.0,9.057040e+06,2627.0,4.362437e+07,643.80
3,2008-03-10,658.290,658.290,658.290,658.290,658.290,18.150,2.840,2.309302e+07,1.458989e+12,1607090.0,7.962789e+10,28694.0,4.667790e+07,1627.0,15561.0,3.327038e+07,2138.0,1.340752e+07,615.71
4,2008-03-11,638.710,638.710,638.710,638.710,638.710,-19.580,-2.970,1.384280e+07,8.374665e+11,226000.0,1.209960e+10,14464.0,1.933131e+07,1337.0,17910.0,2.811427e+07,1570.0,-8.782960e+06,588.26
5,2008-03-12,643.900,643.900,643.900,643.900,643.900,5.190,0.810,1.216691e+07,7.659728e+11,290070.0,2.036917e+10,16836.0,2.378011e+07,1412.0,12032.0,2.034621e+07,1691.0,3.433900e+06,573.45
6,2008-03-13,647.600,647.600,647.600,647.600,647.600,3.700,0.570,8.930090e+06,5.844466e+11,289160.0,1.702626e+10,13298.0,1.703506e+07,1281.0,12256.0,1.900562e+07,1551.0,-1.970560e+06,564.82
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6559,2026-02-20,1829.151,1851.448,1821.603,1849.325,1849.325,28.235,1.551,6.846983e+08,2.152480e+13,24358775.0,8.224590e+11,456926.0,1.195538e+09,2623.9,378513.0,1.167449e+09,3093.7,2.808830e+07,NaN
6562,2026-02-23,1833.900,1863.130,1833.900,1860.140,1860.140,36.050,1.980,7.313611e+08,2.264621e+13,23192200.0,7.774309e+11,486603.0,1.253004e+09,2575.0,401281.0,1.213944e+09,3025.0,3.906029e+07,NaN
6563,2026-02-24,1862.180,1867.690,1849.600,1867.620,1867.620,7.480,0.400,9.667547e+08,3.120594e+13,22413147.0,7.795734e+11,598087.0,1.575227e+09,2634.0,507471.0,1.648273e+09,3248.0,-7.304628e+07,NaN
6564,2026-02-25,1869.490,1876.010,1855.890,1860.910,1860.910,-6.710,-0.360,1.083635e+09,3.567852e+13,62225450.0,1.856894e+12,694266.0,1.862364e+09,2682.0,597096.0,1.892734e+09,3170.0,-3.036983e+07,NaN


### Create features

In [95]:
DF_MAP = {}
DF_MAP

{}

#### add_bbands

In [96]:
DF_MAP[add_bbands.__name__] = {}
vn_index_df_add_bbands = add_bbands(vn_index_df_t2, n=list(range(2, 21)))
DF_MAP[add_bbands.__name__]["dataframe"] = vn_index_df_add_bbands
vn_index_df_add_bbands

,date,open,high,low,close,adjust,change,percent_change,matching_volume,matching_value,...,close_bb_20_bandwidth_slope,close_bb_20_bandwidth_acceleration,close_bb_20_pct_b,close_bb_20_pct_b_slope,close_bb_20_pct_b_gt_1,close_bb_20_pct_b_lt_0,close_bb_20_above_upper,close_bb_20_below_lower,close_bb_20_inside_bands,close_bb_20_position
0,2008-03-07,640.140,640.140,640.140,640.140,640.140,28.970,4.740,8.169220e+06,5.152538e+11,...,NaN,NaN,NaN,NaN,False,False,False,False,True,0
3,2008-03-10,658.290,658.290,658.290,658.290,658.290,18.150,2.840,2.309302e+07,1.458989e+12,...,NaN,NaN,NaN,NaN,False,False,False,False,True,0
4,2008-03-11,638.710,638.710,638.710,638.710,638.710,-19.580,-2.970,1.384280e+07,8.374665e+11,...,NaN,NaN,NaN,NaN,False,False,False,False,True,0
5,2008-03-12,643.900,643.900,643.900,643.900,643.900,5.190,0.810,1.216691e+07,7.659728e+11,...,NaN,NaN,NaN,NaN,False,False,False,False,True,0
6,2008-03-13,647.600,647.600,647.600,647.600,647.600,3.700,0.570,8.930090e+06,5.844466e+11,...,NaN,NaN,NaN,NaN,False,False,False,False,True,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6559,2026-02-20,1829.151,1851.448,1821.603,1849.325,1849.325,28.235,1.551,6.846983e+08,2.152480e+13,...,-0.003677,0.002547,0.818102,0.053949,False,False,False,False,True,0
6562,2026-02-23,1833.900,1863.130,1833.900,1860.140,1860.140,36.050,1.980,7.313611e+08,2.264621e+13,...,0.002366,0.006043,0.887543,0.069440,False,False,False,False,True,0
6563,2026-02-24,1862.180,1867.690,1849.600,1867.620,1867.620,7.480,0.400,9.667547e+08,3.120594e+13,...,0.004535,0.002169,0.905573,0.018031,False,False,False,False,True,0
6564,2026-02-25,1869.490,1876.010,1855.890,1860.910,1860.910,-6.710,-0.360,1.083635e+09,3.567852e+13,...,0.003006,-0.001528,0.819302,-0.086272,False,False,False,False,True,0


#### add_dema

In [97]:
DF_MAP[add_dema.__name__] = {}
vn_index_df_add_dema = add_dema(vn_index_df_t2, n=list(range(2, 21)))
DF_MAP[add_dema.__name__]["dataframe"] = vn_index_df_add_dema
vn_index_df_add_dema

,date,open,high,low,close,adjust,change,percent_change,matching_volume,matching_value,...,close_dema_18_19_direction,close_dema_18_19_dist_slope,close_dema_18_20_dist,close_dema_18_20_dist_abs,close_dema_18_20_direction,close_dema_18_20_dist_slope,close_dema_19_20_dist,close_dema_19_20_dist_abs,close_dema_19_20_direction,close_dema_19_20_dist_slope
0,2008-03-07,640.140,640.140,640.140,640.140,640.140,28.970,4.740,8.169220e+06,5.152538e+11,...,-1,NaN,NaN,NaN,-1,NaN,NaN,NaN,-1,NaN
3,2008-03-10,658.290,658.290,658.290,658.290,658.290,18.150,2.840,2.309302e+07,1.458989e+12,...,-1,NaN,NaN,NaN,-1,NaN,NaN,NaN,-1,NaN
4,2008-03-11,638.710,638.710,638.710,638.710,638.710,-19.580,-2.970,1.384280e+07,8.374665e+11,...,-1,NaN,NaN,NaN,-1,NaN,NaN,NaN,-1,NaN
5,2008-03-12,643.900,643.900,643.900,643.900,643.900,5.190,0.810,1.216691e+07,7.659728e+11,...,-1,NaN,NaN,NaN,-1,NaN,NaN,NaN,-1,NaN
6,2008-03-13,647.600,647.600,647.600,647.600,647.600,3.700,0.570,8.930090e+06,5.844466e+11,...,-1,NaN,NaN,NaN,-1,NaN,NaN,NaN,-1,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6559,2026-02-20,1829.151,1851.448,1821.603,1849.325,1849.325,28.235,1.551,6.846983e+08,2.152480e+13,...,-1,0.279618,-0.325177,0.325177,-1,0.550689,-0.266359,0.266359,-1,0.271071
6562,2026-02-23,1833.900,1863.130,1833.900,1860.140,1860.140,36.050,1.980,7.313611e+08,2.264621e+13,...,1,0.298477,0.263810,0.263810,1,0.588987,0.024151,0.024151,1,0.290510
6563,2026-02-24,1862.180,1867.690,1849.600,1867.620,1867.620,7.480,0.400,9.667547e+08,3.120594e+13,...,1,0.271562,0.803720,0.803720,1,0.539909,0.292498,0.292498,1,0.268347
6564,2026-02-25,1869.490,1876.010,1855.890,1860.910,1860.910,-6.710,-0.360,1.083635e+09,3.567852e+13,...,1,0.109531,1.035922,1.035922,1,0.232202,0.415169,0.415169,1,0.122671


In [98]:
list(DF_MAP.keys())

['add_bbands', 'add_dema']

In [99]:
DF_MAP[add_bbands.__name__]

{'dataframe':             date      open      high       low     close    adjust  change  \
 0     2008-03-07   640.140   640.140   640.140   640.140   640.140  28.970   
 3     2008-03-10   658.290   658.290   658.290   658.290   658.290  18.150   
 4     2008-03-11   638.710   638.710   638.710   638.710   638.710 -19.580   
 5     2008-03-12   643.900   643.900   643.900   643.900   643.900   5.190   
 6     2008-03-13   647.600   647.600   647.600   647.600   647.600   3.700   
 ...          ...       ...       ...       ...       ...       ...     ...   
 6559  2026-02-20  1829.151  1851.448  1821.603  1849.325  1849.325  28.235   
 6562  2026-02-23  1833.900  1863.130  1833.900  1860.140  1860.140  36.050   
 6563  2026-02-24  1862.180  1867.690  1849.600  1867.620  1867.620   7.480   
 6564  2026-02-25  1869.490  1876.010  1855.890  1860.910  1860.910  -6.710   
 6565  2026-02-26  1861.160  1882.170  1859.040  1879.640  1879.640  18.730   
 
       percent_change  matching_volum

## Filter columns

In [100]:
print(
    f"Create feature columns: {[DATE_COLUMN, TARGET_COLUMN, "close", "open", "high", "low", "adjust"]}"
)

for df_name in DF_MAP.keys():
    print(f"\nProcessing dataframe {df_name}")

    DF_MAP[df_name]["feature_columns"] = [
        col
        for col in DF_MAP[df_name]["dataframe"].columns
        if col
        not in [DATE_COLUMN, TARGET_COLUMN, "close", "open", "high", "low", "adjust"]
    ]

Create feature columns: ['date', 'close_5', 'close', 'open', 'high', 'low', 'adjust']

Processing dataframe add_bbands

Processing dataframe add_dema


In [101]:
DF_MAP[add_bbands.__name__]["dataframe"]

,date,open,high,low,close,adjust,change,percent_change,matching_volume,matching_value,...,close_bb_20_bandwidth_slope,close_bb_20_bandwidth_acceleration,close_bb_20_pct_b,close_bb_20_pct_b_slope,close_bb_20_pct_b_gt_1,close_bb_20_pct_b_lt_0,close_bb_20_above_upper,close_bb_20_below_lower,close_bb_20_inside_bands,close_bb_20_position
0,2008-03-07,640.140,640.140,640.140,640.140,640.140,28.970,4.740,8.169220e+06,5.152538e+11,...,NaN,NaN,NaN,NaN,False,False,False,False,True,0
3,2008-03-10,658.290,658.290,658.290,658.290,658.290,18.150,2.840,2.309302e+07,1.458989e+12,...,NaN,NaN,NaN,NaN,False,False,False,False,True,0
4,2008-03-11,638.710,638.710,638.710,638.710,638.710,-19.580,-2.970,1.384280e+07,8.374665e+11,...,NaN,NaN,NaN,NaN,False,False,False,False,True,0
5,2008-03-12,643.900,643.900,643.900,643.900,643.900,5.190,0.810,1.216691e+07,7.659728e+11,...,NaN,NaN,NaN,NaN,False,False,False,False,True,0
6,2008-03-13,647.600,647.600,647.600,647.600,647.600,3.700,0.570,8.930090e+06,5.844466e+11,...,NaN,NaN,NaN,NaN,False,False,False,False,True,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6559,2026-02-20,1829.151,1851.448,1821.603,1849.325,1849.325,28.235,1.551,6.846983e+08,2.152480e+13,...,-0.003677,0.002547,0.818102,0.053949,False,False,False,False,True,0
6562,2026-02-23,1833.900,1863.130,1833.900,1860.140,1860.140,36.050,1.980,7.313611e+08,2.264621e+13,...,0.002366,0.006043,0.887543,0.069440,False,False,False,False,True,0
6563,2026-02-24,1862.180,1867.690,1849.600,1867.620,1867.620,7.480,0.400,9.667547e+08,3.120594e+13,...,0.004535,0.002169,0.905573,0.018031,False,False,False,False,True,0
6564,2026-02-25,1869.490,1876.010,1855.890,1860.910,1860.910,-6.710,-0.360,1.083635e+09,3.567852e+13,...,0.003006,-0.001528,0.819302,-0.086272,False,False,False,False,True,0


## Prepare data

In [102]:
print(f"Create X and y dataframes")

for df_name, cfg in DF_MAP.items():
    print(f"\nProcessing dataframe {df_name}")

    df = cfg["dataframe"]
    df = df.dropna()

    # Ensure datetime + sort
    df[DATE_COLUMN] = pd.to_datetime(df[DATE_COLUMN])
    df = df.sort_values(by=DATE_COLUMN).reset_index(drop=True)

    cfg["dataframe"] = df

    # Train / Validation split
    train_df = df[df[DATE_COLUMN].between(TRAIN_RANGE[0], TRAIN_RANGE[1])]
    val_df = df[df[DATE_COLUMN].between(VALIDATION_RANGE[0], VALIDATION_RANGE[1])]

    cfg["train_dataframe"] = train_df
    cfg["val_dataframe"] = val_df

    # Features / target
    feature_cols = cfg["feature_columns"]

    cfg["X_train"] = train_df[feature_cols]
    cfg["y_train"] = train_df[TARGET_COLUMN]

    cfg["X_val"] = val_df[feature_cols]
    cfg["y_val"] = val_df[TARGET_COLUMN]

Create X and y dataframes

Processing dataframe add_bbands

Processing dataframe add_dema


C:\Users\ADMIN\AppData\Local\Temp\ipykernel_32828\670913343.py:10: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df[DATE_COLUMN] = pd.to_datetime(df[DATE_COLUMN])
C:\Users\ADMIN\AppData\Local\Temp\ipykernel_32828\670913343.py:10: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df[DATE_COLUMN] = pd.to_datetime(df[DATE_COLUMN])


## Model

In [103]:
# Model hyperparameters
MODEL_N_ESTIMATORS = 200  # 5000
MODEL_MAX_DEPTH = 20
MODEL_LEARNING_RATE = 0.01
MODEL_SUBSAMPLE = 0.6
MODEL_COLSAMPLE_BYTREE = 0.5
MODEL_MIN_CHILD_WEIGHT = 30
MODEL_REG_ALPHA = 1.0
MODEL_REG_LAMBDA = 10.0

# Training / system parameters
MODEL_TREE_METHOD = "hist"
MODEL_DEVICE = "cuda"
MODEL_EARLY_STOPPING_ROUNDS = 100
MODEL_ENABLE_CATEGORICAL = True
MODEL_RANDOM_STATE = RANDOM_SEED

N_REPEATS = 1  # 100
TOP_N = 100
CORR_THRESHOLD = 0.95

In [104]:
model = xgb.XGBRegressor(
    n_estimators=MODEL_N_ESTIMATORS,
    max_depth=MODEL_MAX_DEPTH,
    learning_rate=MODEL_LEARNING_RATE,
    subsample=MODEL_SUBSAMPLE,
    colsample_bytree=MODEL_COLSAMPLE_BYTREE,
    min_child_weight=MODEL_MIN_CHILD_WEIGHT,
    reg_alpha=MODEL_REG_ALPHA,
    reg_lambda=MODEL_REG_LAMBDA,
    tree_method=MODEL_TREE_METHOD,
    device=MODEL_DEVICE,
    early_stopping_rounds=MODEL_EARLY_STOPPING_ROUNDS,
    enable_categorical=MODEL_ENABLE_CATEGORICAL,
    random_state=MODEL_RANDOM_STATE,
)

model

,objective,'reg:squarederror'
,base_score,None
,booster,None
,callbacks,None
,colsample_bylevel,None
,colsample_bynode,None
,colsample_bytree,0.5
,device,'cuda'
,early_stopping_rounds,100
,enable_categorical,True
,eval_metric,None


## Feature Importance + Feature Selection + Save Data

In [105]:
FEATURE_SELECTION_RESULT_DIR

'../../src/feature_selection/feature_selection_result'

In [106]:
os.makedirs(FEATURE_SELECTION_RESULT_DIR, exist_ok=True)

In [107]:
for df_name in DF_MAP.keys():
    print(f"\n{'═' * 60}")
    print(f"  DATAFRAME: {df_name}")
    print(f"{'═' * 60}")

    out_dir = os.path.join(FEATURE_SELECTION_RESULT_DIR, df_name)
    os.makedirs(out_dir, exist_ok=True)

    cfg = DF_MAP[df_name]
    df = cfg["dataframe"]

    model_copy = clone(model)
    model_copy.fit(
        cfg["X_train"],
        cfg["y_train"],
        eval_set=[(cfg["X_val"], cfg["y_val"])],
        verbose=100,
    )

    X = df[cfg["feature_columns"]]
    y = df[TARGET_COLUMN]

    # ── BASELINE ────────────────────────────────────────────────
    print("\n[1/7] Baseline prediction (GPU)...")
    baseline_pred = model_copy.predict(X)
    baseline_mse = np.mean((y - baseline_pred) ** 2)
    print(f"      ✔ Baseline MSE : {baseline_mse:.6f}")

    # ── PERMUTATION IMPORTANCE ──────────────────────────────────
    print("\n[2/7] Permutation importance (GPU)...")
    X_vals = X.values
    y_vals = y.values
    importances = []

    for i, col in enumerate(tqdm(X.columns, desc="Features", ncols=70)):
        scores = np.empty(N_REPEATS)
        col_backup = X_vals[:, i].copy()

        for r in range(N_REPEATS):
            X_vals[:, i] = np.random.permutation(col_backup)
            pred = model_copy.predict(X_vals)
            scores[r] = np.mean((y_vals - pred) ** 2)

        X_vals[:, i] = col_backup
        importances.append(scores.mean() - baseline_mse)

    # ── IMPORTANCE DATAFRAME ────────────────────────────────────
    print("\n[3/7] Building importance dataframe...")
    importance_df = (
        pd.DataFrame({"feature": X.columns, "importance": importances})
        .sort_values("importance", ascending=False)
        .reset_index(drop=True)
    )
    importance_df["importance_pct"] = (
        importance_df["importance"] / importance_df["importance"].sum() * 100
    )
    importance_df.to_csv(
        os.path.join(out_dir, f"{df_name}_importance_before.csv"), index=False
    )
    print(f"      ✔ Saved → {df_name}_importance_before.csv")

    # ── HELPERS ─────────────────────────────────────────────────
    def plot_top_n(imp_df, title, filename):
        top = (
            imp_df.sort_values("importance", ascending=False)
            .head(TOP_N)
            .set_index("feature")["importance"]
            .sort_values(ascending=True)
        )
        ax = top.plot(kind="barh", figsize=(12, 12))
        plt.title(title)
        plt.ylabel("Importance")
        ax.xaxis.set_ticks_position("both")
        ax.tick_params(axis="x", which="both", top=True, bottom=True, labeltop=True)
        plt.tight_layout()
        plt.savefig(os.path.join(out_dir, filename), dpi=400, bbox_inches="tight")
        plt.close()

    def plot_corr(features, title, filename):
        corr = df[features].corr()
        plt.figure(figsize=(14, 12))
        sns.heatmap(corr, cmap="coolwarm", center=0, square=True, linewidths=0.5)
        plt.title(title)
        plt.tight_layout()
        plt.savefig(os.path.join(out_dir, filename), dpi=400, bbox_inches="tight")
        plt.close()

    # ── PLOTS BEFORE REMOVAL ────────────────────────────────────
    print("\n[4/7] Plotting top-N importance (before removal)...")
    plot_top_n(
        importance_df,
        f"Top {TOP_N} Feature Importance (Before)",
        "feature_importance_plot_before.png",
    )
    print(f"      ✔ Saved → feature_importance_plot_before.png")

    print("\n[5/7] Plotting correlation heatmap (before removal)...")
    top_features_before = importance_df.head(TOP_N)["feature"].tolist()
    plot_corr(
        top_features_before,
        "Feature Correlation Matrix (Before)",
        "feature_correlation_plot_before.png",
    )
    print(f"      ✔ Saved → feature_correlation_plot_before.png")

    # ── REMOVE CORRELATED FEATURES ──────────────────────────────
    print("\n[6/7] Removing highly correlated features...")
    corr_matrix_abs = df[cfg["feature_columns"]].corr().abs()
    corr_matrix_abs.to_csv(os.path.join(out_dir, f"{df_name}_corr_matrix_abs.csv"))
    sorted_features = importance_df["feature"].tolist()

    selected_features, removed_features = [], set()
    for feature in sorted_features:
        if feature in removed_features:
            continue
        selected_features.append(feature)
        correlated = corr_matrix_abs.index[
            corr_matrix_abs[feature] > CORR_THRESHOLD
        ].tolist()
        removed_features.update(f for f in correlated if f != feature)

    print(f"      ┌─────────────────────────────────")
    print(f"      │ Threshold  : {CORR_THRESHOLD}")
    print(f"      │ Total      : {len(sorted_features)}")
    print(f"      │ Selected   : {len(selected_features)}")
    print(f"      │ Removed    : {len(removed_features)}")
    print(f"      └─────────────────────────────────")
    print(f"      Top kept : {selected_features[:5]}")
    print(
        f"      Removed  : {list(removed_features)[:5]}{'...' if len(removed_features) > 5 else ''}"
    )

    # ── PLOTS AFTER REMOVAL ─────────────────────────────────────
    print("\n[7/7] Plotting top-N importance & correlation (after removal)...")
    importance_df_after = importance_df[
        importance_df["feature"].isin(selected_features)
    ].reset_index(drop=True)
    importance_df_after.to_csv(
        os.path.join(out_dir, f"{df_name}_importance_after.csv"), index=False
    )
    plot_top_n(
        importance_df_after,
        f"Top {TOP_N} Feature Importance (After)",
        "feature_importance_plot_after.png",
    )
    top_features_after = importance_df_after.head(TOP_N)["feature"].tolist()
    plot_corr(
        top_features_after,
        "Feature Correlation Matrix (After)",
        "feature_correlation_plot_after.png",
    )
    print(f"      ✔ Saved → {df_name}_importance_after.csv")
    print(f"      ✔ Saved → feature_importance_plot_after.png")
    print(f"      ✔ Saved → feature_correlation_plot_after.png")

    print(f"\n{'─' * 60}")
    print(f"  ✅ Done: {df_name}")
    print(f"{'─' * 60}\n")


════════════════════════════════════════════════════════════
  DATAFRAME: add_bbands
════════════════════════════════════════════════════════════
[0]	validation_0-rmse:519.72552
[100]	validation_0-rmse:214.16347
[199]	validation_0-rmse:97.86018

[1/7] Baseline prediction (GPU)...
      ✔ Baseline MSE : 9931.943826

[2/7] Permutation importance (GPU)...


Features: 100%|█████████████████████| 450/450 [00:44<00:00, 10.09it/s]



[3/7] Building importance dataframe...
      ✔ Saved → add_bbands_importance_before.csv

[4/7] Plotting top-N importance (before removal)...
      ✔ Saved → feature_importance_plot_before.png

[5/7] Plotting correlation heatmap (before removal)...
      ✔ Saved → feature_correlation_plot_before.png

[6/7] Removing highly correlated features...
      ┌─────────────────────────────────
      │ Threshold  : 0.95
      │ Total      : 450
      │ Selected   : 281
      │ Removed    : 169
      └─────────────────────────────────
      Top kept : ['close_bb_3_lower', 'matching_value', 'number_of_sell_orders', 'buy_volume', 'close_bb_6_dist_upper']
      Removed  : ['close_bb_9_lower', 'close_bb_5_dist_lower', 'close_bb_8_pct_b', 'close_bb_11_middle', 'close_bb_15_pct_b']...

[7/7] Plotting top-N importance & correlation (after removal)...
      ✔ Saved → add_bbands_importance_after.csv
      ✔ Saved → feature_importance_plot_after.png
      ✔ Saved → feature_correlation_plot_after.png

─────

Features: 100%|█████████████████████| 811/811 [02:10<00:00,  6.20it/s]



[3/7] Building importance dataframe...
      ✔ Saved → add_dema_importance_before.csv

[4/7] Plotting top-N importance (before removal)...
      ✔ Saved → feature_importance_plot_before.png

[5/7] Plotting correlation heatmap (before removal)...
      ✔ Saved → feature_correlation_plot_before.png

[6/7] Removing highly correlated features...
      ┌─────────────────────────────────
      │ Threshold  : 0.95
      │ Total      : 811
      │ Selected   : 110
      │ Removed    : 701
      └─────────────────────────────────
      Top kept : ['close_dema_2', 'number_of_sell_orders', 'matching_value', 'buy_volume', 'close_dema_10_slope']
      Removed  : ['close_dema_11_15_dist_abs', 'close_dema_13_16_dist', 'close_dema_5_19_dist_abs', 'close_dema_8_dist_abs', 'close_dema_11_14_dist_slope']...

[7/7] Plotting top-N importance & correlation (after removal)...
      ✔ Saved → add_dema_importance_after.csv
      ✔ Saved → feature_importance_plot_after.png
      ✔ Saved → feature_correlation_p